# Source landscape — who feeds the machine?

Each row carries a `source`. This notebook maps the ecosystem of sources:
their sheer volume, their vocabulary breadth, and how their share of the
stream shifts over time.

*SQL is kept inline in this notebook.*

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Connection settings come from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query and return a DataFrame. Params are passed safely to psycopg."""
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Ready — connected settings for", os.environ["POSTGRES_HOST"])


## Volume vs. vocabulary breadth
A source can be loud (many mentions) or diverse (many distinct phrases).
The treemap area is total mentions; color is how many *distinct* phrases
the source contributes.

In [ ]:
sources = q("""
    SELECT source,
           COUNT(*)               AS mentions,
           COUNT(DISTINCT words)  AS distinct_phrases
    FROM ngrams
    GROUP BY source
    ORDER BY mentions DESC
""")

fig = px.treemap(
    sources,
    path=[px.Constant("all sources"), "source"],
    values="mentions",
    color="distinct_phrases",
    color_continuous_scale="Viridis",
    title="Sources by volume (area) and vocabulary breadth (color)",
)
fig.update_layout(height=600, margin=dict(t=50, l=10, r=10, b=10))
fig.show()

## Shifting share of voice
Daily volume of the top sources as a stacked area — who dominated when.

In [ ]:
daily = q("""
    WITH top AS (
        SELECT source
        FROM ngrams
        GROUP BY source
        ORDER BY COUNT(*) DESC
        LIMIT 8
    )
    SELECT date_trunc('day', timestamp) AS day,
           source,
           COUNT(*)                     AS n
    FROM ngrams
    WHERE source IN (SELECT source FROM top)
    GROUP BY 1, 2
    ORDER BY 1
""")

fig = px.area(
    daily, x="day", y="n", color="source",
    title="Daily volume of the top 8 sources",
)
fig.update_layout(height=480, xaxis_title="", yaxis_title="occurrences")
fig.show()